# LightRAG — What It Is, How It Works, and a Working Example

**Paper:** [LightRAG: Simple and Fast Retrieval-Augmented Generation](https://arxiv.org/abs/2410.05779) (EMNLP 2025)  
**GitHub:** [HKUDS/LightRAG](https://github.com/HKUDS/LightRAG) (30K+ stars)  
**Package:** `pip install lightrag-hku`

---
## What Problem Does LightRAG Solve?

Microsoft's **GraphRAG** is powerful but has two big problems:

1. **Extremely expensive** — it runs Leiden community detection, then generates LLM-written summary reports for each community. For a large corpus this costs ~14 million tokens just for indexing.

2. **Can't do incremental updates** — when you add new documents, you must tear down and rebuild the entire community structure from scratch.

**LightRAG's insight:** Skip community detection entirely. Instead, use the entity-relationship graph directly with smart key-value indexing.

```
GraphRAG pipeline:                         LightRAG pipeline:
  Documents                                  Documents
     │                                          │
     ▼                                          ▼
  Chunk text                                 Chunk text
     │                                          │
     ▼                                          ▼
  Extract entities+relations                 Extract entities+relations
     │                                          │
     ▼                                          ▼
  Leiden community detection  ← EXPENSIVE    Deduplicate + profile    ← CHEAP
     │                                          │
     ▼                                          ▼
  Generate community reports  ← EXPENSIVE    Embed descriptions       ← CHEAP
     │                                          │
     ▼                                          ▼
  Search communities                         Dual-mode retrieval
                                             (local + global + hybrid)
```

---
## How LightRAG Works Internally

### Indexing (one-time per document)

1. **Chunk** the document (default 1200 tokens, 100 overlap)
2. **Extract** entities and relations from each chunk using LLM
3. **Profile** each entity/relation — generate key-value descriptions:
   - Entity keys = entity names
   - Relation keys = LLM-enhanced keywords enriched with **global themes** from connected entities
4. **Deduplicate** — merge identical entities across chunks
5. **Embed** all descriptions into vectors

### Retrieval (per query)

Four modes available:

| Mode | What It Searches | Best For |
|------|-----------------|----------|
| `naive` | Raw text chunks (standard RAG) | Simple fact lookup |
| `local` | Specific entities + their neighbors | "What does X do?" |
| `global` | Relation-level themes across entities | "What are the main themes?" |
| `hybrid` | Both local + global combined | Best overall quality |

### Why No Community Detection?

GraphRAG uses communities to answer global questions ("What are the themes?"). LightRAG replaces this with **relation-level keys enhanced by connected entity themes**. When you extract a relation like `(Einstein)--[DEVELOPED]-->(Relativity)`, LightRAG's profiling step adds global context: this relation connects to themes of "physics", "theoretical science", "Nobel Prize". These enhanced keys let global retrieval work without expensive community summaries.

---
## Cost Comparison

| | GraphRAG | LightRAG |
|--|---------|----------|
| Indexing cost (large corpus) | ~$33K+ | **~$0.50** |
| Community detection | Yes (expensive) | **No** |
| Community report generation | ~14M tokens | **0 tokens** |
| Incremental updates | Rebuild everything | **Just add new entities** |
| Query latency | ~2-5s | **~80ms** |
| Quality vs GraphRAG | 100% (baseline) | **70-90%** |

LightRAG trades ~10-30% quality for 100x cost reduction and instant incremental updates.

---
## Benchmark Results (from the paper)

Evaluated on Agriculture, CS, Legal, Mixed datasets. Win rates vs NaiveRAG:

| Dataset | Comprehensiveness | Diversity | Overall |
|---------|------------------|-----------|--------|
| Agriculture | 67.6% | 76.4% | 67.6% |
| CS | 61.6% | 62.0% | 61.2% |
| Legal | 83.6% | 86.4% | 84.8% |
| Mixed | 61.2% | 67.6% | ~61% |

LightRAG also beat GraphRAG, RQ-RAG, and HyDE on all datasets.

---
## Supported Backends

LightRAG is backend-agnostic:

| Component | Default | Options |
|-----------|---------|--------|
| **Graph** | NetworkX (file-based) | Neo4j, PostgreSQL, Apache AGE |
| **Vector** | NanoVectorDB (file-based) | Milvus, Chroma, Faiss, Qdrant, pgvector |
| **KV Store** | JSON files | Redis, MongoDB, PostgreSQL |
| **LLM** | OpenAI GPT-4o-mini | Anthropic, Ollama, HuggingFace, any OpenAI-compatible |

---
## Working Example

Let's run LightRAG on our Meridian document and compare with our manual Graph RAG.

In [ ]:
# Install LightRAG
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "lightrag-hku", "-q"])
print("LightRAG installed!")

In [ ]:
import os, asyncio
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

# Verify API key is set
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
print(f"API key set: {os.environ['OPENAI_API_KEY'][:10]}...")

In [ ]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import gpt_4o_mini_complete, openai_embed

WORKING_DIR = "./lightrag_storage"
os.makedirs(WORKING_DIR, exist_ok=True)

# Initialize LightRAG — this is the entire setup
rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=gpt_4o_mini_complete,
    embedding_func=openai_embed,
    # All defaults: NetworkX graph, NanoVectorDB, JSON KV store
)

# REQUIRED: initialize storage backends
await rag.initialize_storages()

print("LightRAG initialized!")
print(f"Storage dir: {WORKING_DIR}")

### Insert the document

LightRAG handles chunking, entity extraction, deduplication, and embedding automatically.

In [ ]:
document = Path("../data/meridian_dossier.txt").read_text()
print(f"Inserting document: {len(document)} chars, {len(document.split())} words")
print("This will chunk → extract entities/relations → embed. Takes ~30-60 seconds...\n")

await rag.ainsert(document)

print("\nDocument indexed!")

### Query in all four modes

In [ ]:
async def query_all_modes(question):
    print(f"Question: {question}\n")
    for mode in ["naive", "local", "global", "hybrid"]:
        result = await rag.aquery(question, param=QueryParam(mode=mode))
        print(f"[{mode.upper():6}] {result[:300]}")
        print()

await query_all_modes("Who founded Meridian and when?")

In [ ]:
await query_all_modes("List every person who left Meridian and where they went.")

In [ ]:
await query_all_modes("What path connects Thomas Lee to the Illumina acquisition?")

In [ ]:
await query_all_modes("Trace Priya Mehta's ironic journey from Meridian champion to BioNexus board member.")

In [ ]:
await query_all_modes("List ALL funding rounds with exact amounts and investors.")

### Inspect the extracted graph

In [ ]:
# LightRAG stores the graph as a NetworkX file
import networkx as nx

graph_file = Path(WORKING_DIR) / "graph_chunk_entity_relation.graphml"
if graph_file.exists():
    G = nx.read_graphml(str(graph_file))
    print(f"LightRAG extracted graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"\nTop 15 entities by connections:")
    for name, deg in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:15]:
        print(f"  {name}: {deg} connections")
else:
    print("Graph file not found — check working_dir")
    # Try listing what's in the directory
    for f in Path(WORKING_DIR).iterdir():
        print(f"  {f.name} ({f.stat().st_size} bytes)")

In [ ]:
# Visualize LightRAG's graph
from pyvis.network import Network

if graph_file.exists():
    net = Network(height="800px", width="100%", directed=True,
                  bgcolor="#0d1117", font_color="white", cdn_resources="remote")
    net.barnes_hut(gravity=-5000, spring_length=200)
    
    for node in G.nodes():
        net.add_node(node, label=node[:25], title=node, color="#4ECDC4", size=15)
    for src, tgt in G.edges():
        net.add_edge(src, tgt, color="#636e72")
    
    html_path = os.path.abspath("lightrag_graph.html")
    net.save_graph(html_path)
    os.system(f"open '{html_path}'")
    print(f"Graph visualization: {html_path}")

### Incremental update — add a new document

This is LightRAG's killer feature over GraphRAG — no rebuild needed.

In [ ]:
new_doc = """
BREAKING NEWS (March 2026): Meridian Health Technologies has officially filed for an IPO on the
New York Stock Exchange, targeting a $2.5 billion valuation. Goldman Sachs and Morgan Stanley are
serving as lead underwriters. CEO Dr. Fatima Al-Hassan stated that the company's annual recurring
revenue has reached $140M, up from $85M a year earlier. The company plans to use IPO proceeds to
fund expansion into Asia-Pacific markets, with a new office planned in Singapore. Former CTO
Lars Eriksson, now at Recursion Pharmaceuticals, congratulated the team via LinkedIn.
"""

print("Inserting new document (incremental update — no rebuild!)...")
await rag.ainsert(new_doc)
print("Done! Graph updated seamlessly.\n")

# Now query about the new info
result = await rag.aquery(
    "What are Meridian's IPO plans? What is their current revenue?",
    param=QueryParam(mode="hybrid")
)
print(f"Answer: {result}")

In [ ]:
# Clean up
await rag.finalize_storages()
print("LightRAG storage finalized.")

---
## Summary: LightRAG vs GraphRAG vs Our Manual Approach

| Aspect | Our Manual KG | Microsoft GraphRAG | LightRAG |
|--------|-------------|-------------------|----------|
| **Extraction** | We wrote the code | Built-in pipeline | Built-in pipeline |
| **Graph storage** | Neo4j | Custom | NetworkX/Neo4j/Postgres |
| **Community detection** | No | Yes (Leiden) | **No** |
| **Query modes** | Custom Cypher | Local + Global + DRIFT | Naive + Local + Global + Hybrid |
| **Incremental updates** | Manual MERGE | Full rebuild | **Seamless** |
| **Cost** | Low (our LLM calls) | Very high | **Very low** |
| **Lines of code** | ~100 | pip install | pip install |
| **Control** | Full control | Configuration | Configuration |

### When to use what?

- **Manual KG (our approach):** When you need full control over schema, extraction, and queries. Best for learning and custom use cases.
- **LightRAG:** When you want Graph RAG with minimal code, low cost, and incremental updates. Best production choice for most teams.
- **GraphRAG:** When you need the absolute highest quality global summarization and have budget for it.